#### [ CNN CUSTOM DATASET + MODEL ]

- 사용자 정의 이미지 데이터셋 생성 ==> ImageFolder 사용
- 사용자 정의 cnn기반 모델 설계
- 데이터 : 강아지, 고양이 사진



[1] 모듈 로딩 및 데이터 준비 <hr>

In [1]:
## 모듈 로딩
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.utils
import torch.utils.data

from torchvision.datasets import ImageFolder            ## 이미지용 데이터셋 생성 모듈
from torch.utils.data import DataLoader                 ## 데이터로더
from torchvision.transforms import transforms           ## 이미지 전처리 및 증강 모듈


import matplotlib.pyplot as plt


In [2]:
## 데이터 준비
IMG_ROOT = '../_image/cat_dog/'
IMG_TRAIN_ROOT = '../_image/cat_dog_train/'

[2] 데이터 로딩 및 데이터셋 준비 <hr>

In [3]:
# print(f"classes  => {imgDS.classes}")
# print(f"class_to_idx  => {imgDS.class_to_idx}")
# print(f"targets  => {imgDS.targets}")
# print(f"imgs  => {imgDS.imgs}")
# print(imgDS[0])

In [4]:
from PIL import Image

In [ ]:
## 2-0 이미지 전처리 및 변형
## resize - > 이미지 크기 통일      ==> transforms.Resize((shape))
## tensor --> 텐서로 타입 변형.     ==> transforms.ToTensor() : 텐서화 + 정규화(0~1)
##                                  ==>                         채널(C, H, W)로 변경.
preprocessing = transforms.Compose(
    [
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((50,50)),
        transforms.ToTensor()
    ]
    )

testDS = ImageFolder(IMG_ROOT, transform=preprocessing)
trainDS = ImageFolder(IMG_TRAIN_ROOT, transform=preprocessing)


In [6]:
testDS

Dataset ImageFolder
    Number of datapoints: 138
    Root location: ../_image/cat_dog/
    StandardTransform
Transform: Compose(
               Resize(size=(50, 50), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )

In [7]:
trainDS

Dataset ImageFolder
    Number of datapoints: 712
    Root location: ../_image/cat_dog_train/
    StandardTransform
Transform: Compose(
               Resize(size=(50, 50), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           )

In [8]:
### CNN으로 캣/독 이진분류.

class CD_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        ## 특징맵 추출 부분
        self.con_layer1 = nn.Conv2d(3, 16, 3, padding=1)  # (1, 3, 50, 50) -> (1, 16, 50, 50)
        self.pool_layer1 = nn.MaxPool2d(2, 2)             # (1, 16, 50, 50) -> (1, 16, 50, 50)
        self.con_layer2 = nn.Conv2d(16, 32, 3, padding=1) # (1, 16, 50, 50) -> (1, 32, 50, 50)
        self.pool_layer2 = nn.MaxPool2d(2, 2)             # (1, 32, 50, 50) -> (1, 32, 25, 25)
        
        self.flat_layer = nn.Flatten()                    # (1, 32, 25, 25) -> (1, 32*25*25)
        
        ## 전결합 학습 부분
        self.fc_layer1 = nn.Linear(32*12*12, 512)
        self.h_layer1 = nn.Linear(512, 128)
        self.drop = nn.Dropout(0.25)  # Dropout2d 대신 Dropout 사용
        self.h_layer2 = nn.Linear(128, 64)
        self.h_layer3 = nn.Linear(64, 32)
        self.out_layer = nn.Linear(32, 1)  # 이진 분류 (출력 노드 1개)
        
    def forward(self, x):
        x = self.pool_layer1(nn.ReLU()(self.con_layer1(x)))
        x = self.pool_layer2(nn.ReLU()(self.con_layer2(x)))
        x = self.flat_layer(x)
        x = nn.ReLU()(self.fc_layer1(x))
        x = nn.ReLU()(self.h_layer1(x))
        x = self.drop(x)
        x = nn.ReLU()(self.h_layer2(x))
        x = nn.ReLU()(self.h_layer3(x))
        x = self.out_layer(x)  
        return x
        



In [9]:
test = CD_CNN()

In [10]:
import sys

sys.path.append('../_utils/')
from DL_Module import TT_classifier

In [11]:
testCF = TT_classifier(test, trainDS, testDS, LOSS_FN=nn.BCEWithLogitsLoss())

EPOCHS : 100
ITERATION : 7
LOSS_FN : BCEWithLogitsLoss()


In [12]:
len(testDS)
for a, b in testDS:
    print( a.shape, b)
    break

torch.Size([3, 50, 50]) 0


In [13]:
# TRAINDL   = DataLoader(trainDS, batch_size=100) ## 학습용 데이터로더

In [14]:
# for feature, target in TRAINDL:
#     print(test(feature).shape)
#     print( target.reshape(-1,1).shape)
#     break

In [ ]:
HIST = testCF.cycling()

torch.Size([138, 3, 50, 50])
torch.Size([138])

EPOCH[0/100]----------------
- TRAIN_LOSS 0.79356  ACC 0.60000
- VALID_LOSS 0.68598  ACC 0.43478
[0] - num_bad_epochs : 0 
torch.Size([138, 3, 50, 50])
torch.Size([138])

EPOCH[1/100]----------------
- TRAIN_LOSS 0.78900  ACC 0.60048
- VALID_LOSS 0.67666  ACC 0.54348
[1] - num_bad_epochs : 0 
torch.Size([138, 3, 50, 50])
torch.Size([138])

EPOCH[2/100]----------------
- TRAIN_LOSS 0.76083  ACC 0.71714
- VALID_LOSS 0.66449  ACC 0.60870
[2] - num_bad_epochs : 0 
torch.Size([138, 3, 50, 50])
torch.Size([138])

EPOCH[3/100]----------------
- TRAIN_LOSS 0.70657  ACC 0.76762
- VALID_LOSS 0.71463  ACC 0.56522
[3] - num_bad_epochs : 1 
torch.Size([138, 3, 50, 50])
torch.Size([138])

EPOCH[4/100]----------------
- TRAIN_LOSS 0.76604  ACC 0.66714
- VALID_LOSS 0.70855  ACC 0.51449
[4] - num_bad_epochs : 2 
torch.Size([138, 3, 50, 50])
torch.Size([138])

EPOCH[5/100]----------------
- TRAIN_LOSS 0.72539  ACC 0.72952
- VALID_LOSS 0.67165  ACC 0.59420


In [ ]:
testCF.draw_graph()

ValueError: too many values to unpack (expected 2)